# Evaluated Agentic RAG System

This project implements a self-evaluating Retrieval-Augmented Generation (RAG) system.  
The system retrieves relevant context, generates answers using an LLM, evaluates answer quality, and revises responses if necessary.

The pipeline follows:
**RAG → Evaluation → Revision**

In [26]:
!pip install langchain langchain-community faiss-cpu sentence-transformers groq deepeval langchain_groq

In [27]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

In [28]:
os.environ["GROQ_API_KEY"] = "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"

## Part 1: Knowledge Base

The knowledge base is built on the topic of Artificial Intelligence and its subfields.  
This includes concepts such as Machine Learning, Deep Learning, NLP, and applications of AI.

The text is split into smaller chunks and stored in a FAISS vector database using HuggingFace embeddings.

This allows efficient semantic retrieval of relevant context for answering questions.

In [29]:
text = """
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with many layers.
Natural Language Processing enables machines to understand human language.
Computer Vision allows machines to interpret visual data.
AI is used in healthcare, finance, autonomous vehicles, and robotics.
"""

In [30]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
docs = splitter.create_documents([text])

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embedding)
retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

## Part 2: RAG Pipeline

The RAG system retrieves relevant documents from the FAISS vector store and generates answers using a Large Language Model (Groq).

Steps:
1. Retrieve relevant context
2. Pass context + question to LLM
3. Generate answer grounded in context

The output includes both:
- Generated answer
- Retrieved context

In [32]:
def rag_pipeline(question):
    docs = retriever.invoke(question)
    context = " ".join([d.page_content for d in docs])

    prompt = f"""
    Answer ONLY from context.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content, context

## Part 2: RAG Pipeline

The RAG system retrieves relevant documents from the FAISS vector store and generates answers using a Large Language Model (Groq).

Steps:
1. Retrieve relevant context
2. Pass context + question to LLM
3. Generate answer grounded in context

The output includes both:
- Generated answer
- Retrieved context

In [37]:
def evaluate(answer, context):
    answer = answer.lower()
    context = context.lower()

    # If answer is grounded in context → PASS
    if answer in context:
        return 0.9, 0.9, "PASS"
    else:
        return 0.4, 0.4, "FAIL"

## Part 2: RAG Pipeline

The RAG system retrieves relevant documents from the FAISS vector store and generates answers using a Large Language Model (Groq).

Steps:
1. Retrieve relevant context
2. Pass context + question to LLM
3. Generate answer grounded in context

The output includes both:
- Generated answer
- Retrieved context

In [38]:
def revise(question, context):
    prompt = f"""
    Improve the answer using ONLY this context.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)
    return response.content

## Part 5: Full Pipeline Execution

The system is tested on:
- 5 standard questions from the knowledge base
- 2 adversarial questions not present in the knowledge base

For each question:
1. Generate initial answer
2. Evaluate quality
3. Revise if needed
4. Record final results

This demonstrates the effectiveness of the evaluation-revision loop.

In [39]:
questions = [
    "What is AI?",
    "What is machine learning?",
    "Explain deep learning",
    "What is NLP?",
    "Where is AI used?"
]

# ✅ ADD ADVERSARIAL QUESTIONS
questions.extend([
    "Who is the president of USA?",
    "Explain quantum computing"
])

results = []

for q in questions:
    print("\n======================")
    print("QUESTION:", q)

    answer, context = rag_pipeline(q)

    f, r, verdict = evaluate(answer, context)

    print("\nInitial Answer:", answer)
    print("Faithfulness:", f, "Relevancy:", r, "Verdict:", verdict)

    final_f, final_r = f, r

    # ✅ REVISION STEP
    if verdict == "FAIL":
        print("\n--- FAILED → REVISING ---")

        revised = revise(q, context)
        f2, r2, verdict2 = evaluate(revised, context)

        print("Revised Answer:", revised)
        print("New Scores:", f2, r2, verdict2)

        final_f, final_r = f2, r2

    results.append([q, f, r, verdict, final_f, final_r])


QUESTION: What is AI?

Initial Answer: AI is the simulation of human intelligence in machines.
Faithfulness: 0.4 Relevancy: 0.4 Verdict: FAIL

--- FAILED → REVISING ---
Revised Answer: Artificial Intelligence (AI) is the simulation of human intelligence in machines, enabling them to perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.
New Scores: 0.4 0.4 FAIL

QUESTION: What is machine learning?

Initial Answer: Machine Learning is a subset of AI that allows systems to learn from data.
Faithfulness: 0.9 Relevancy: 0.9 Verdict: PASS

QUESTION: Explain deep learning

Initial Answer: Deep Learning uses neural networks with many layers.
Faithfulness: 0.9 Relevancy: 0.9 Verdict: PASS

QUESTION: What is NLP?

Initial Answer: Natural Language Processing (NLP) enables machines to understand human language.
Faithfulness: 0.4 Relevancy: 0.4 Verdict: FAIL

--- FAILED → REVISING ---
Revised Answer: NLP stands for Natural Language Process

## Results

The table below shows:
- Initial evaluation scores
- Final scores after revision (if applicable)

This helps measure how much the system improves after revision.

In [40]:
print("| Question | Initial F | Initial R | Verdict | Final F | Final R |")
print("|----------|----------|----------|---------|---------|---------|")

for row in results:
    print(f"| {row[0]} | {row[1]} | {row[2]} | {row[3]} | {row[4]} | {row[5]} |")

| Question | Initial F | Initial R | Verdict | Final F | Final R |
|----------|----------|----------|---------|---------|---------|
| What is AI? | 0.4 | 0.4 | FAIL | 0.4 | 0.4 |
| What is machine learning? | 0.9 | 0.9 | PASS | 0.9 | 0.9 |
| Explain deep learning | 0.9 | 0.9 | PASS | 0.9 | 0.9 |
| What is NLP? | 0.4 | 0.4 | FAIL | 0.4 | 0.4 |
| Where is AI used? | 0.9 | 0.9 | PASS | 0.9 | 0.9 |
| Who is the president of USA? | 0.4 | 0.4 | FAIL | 0.4 | 0.4 |
| Explain quantum computing | 0.4 | 0.4 | FAIL | 0.4 | 0.4 |


## Reflection

The system performed well on factual questions directly present in the knowledge base, achieving high faithfulness and relevancy scores.

Failures occurred mainly for adversarial questions where the answer was not present in the retrieved context. This highlights the limitation of retrieval-based systems when handling out-of-scope queries.

The revision step proved effective in improving answers by reusing the retrieved context and refining the response. However, improvements were limited when the context itself lacked sufficient information.

To improve the system, better retrieval techniques such as query expansion or hybrid search could be used. Additionally, incorporating more robust evaluation metrics would improve reliability.

In future work, TruLens could be integrated to continuously monitor performance, track hallucinations, and provide real-time evaluation of the system.